# Transaction risk: data, features and models

What this notebook is for: showing what the simulated fraud actually looks like,
why the feature design is what it is, and where the models' performance comes
from. It runs offline on a small configuration in about a minute.

The published results are in `docs/results.md`, produced by `scripts/train.py`
at the default configuration. Nothing here supersedes them.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from rtml.config import PipelineConfig
from rtml.data.simulator import SimulationConfig, simulate
from rtml.features.engineering import build_features, feature_names
from rtml.logging_utils import configure_logging
from rtml.training.pipeline import train_all

configure_logging("WARNING")
pd.set_option("display.width", 120)

ROOT = Path.cwd().parent
config = PipelineConfig.from_yaml(ROOT / "configs" / "fast.yaml").with_overrides({
    "simulation.n_customers": 800,
    "simulation.n_terminals": 1600,
    "simulation.n_days": 80,
    "split.train_days": 49,
    "split.delay_days": 7,
    "split.test_days": 17,
    "split.validation_days": 12,
    "evaluation.bootstrap_resamples": 200,
})
result = simulate(SimulationConfig(**config.simulation.model_dump(), seed=config.run.seed))
tx = result.transactions
print(f"{len(tx):,} transactions, fraud rate {tx['is_fraud'].mean():.3%}")

127,213 transactions, fraud rate 4.452%


## 1. The three fraud patterns

Each scenario models a different compromise, and each needs different features
to detect. A pipeline computing only transaction-level features catches
scenario 1 and nothing else — and its aggregate score still looks respectable,
which is why per-scenario recall is reported separately in `docs/results.md`.

In [2]:
fraud = tx[tx.is_fraud == 1]
summary = fraud.groupby("fraud_scenario").agg(
    cases=("transaction_id", "size"),
    mean_amount=("tx_amount", "mean"),
    median_amount=("tx_amount", "median"),
    distinct_terminals=("terminal_id", "nunique"),
    distinct_customers=("customer_id", "nunique"),
)
summary["share_of_fraud"] = (summary["cases"] / len(fraud)).round(3)
print(f"legitimate mean amount: {tx[tx.is_fraud == 0]['tx_amount'].mean():.2f}\n")
summary.round(2)

legitimate mean amount: 53.58



,cases,mean_amount,median_amount,distinct_terminals,distinct_customers,share_of_fraud
fraud_scenario,,,,,,
1,87,235.40,231.76,81,50,0.02
2,3766,54.58,46.36,148,510,0.66
3,1810,265.72,228.00,760,188,0.32


Scenario 1 is separable on amount alone. Scenario 3 inflates a customer's own
amounts fivefold, so it is only visible *relative to that customer's baseline* —
which is exactly what the customer average-amount features provide. Scenario 2
leaves the amount untouched entirely: nothing about the transaction itself is
unusual, and only the terminal's recent history gives it away.

In [3]:
# Scenario 2 is invisible at transaction level - the amounts look ordinary.
legit = tx[tx.is_fraud == 0]["tx_amount"]
for scenario in (1, 2, 3):
    amounts = fraud[fraud.fraud_scenario == scenario]["tx_amount"]
    print(f"scenario {scenario}: mean {amounts.mean():7.2f}  "
          f"vs legitimate {legit.mean():7.2f}  "
          f"(ratio {amounts.mean()/legit.mean():.2f}x)")

scenario 1: mean  235.40  vs legitimate   53.58  (ratio 4.39x)
scenario 2: mean   54.58  vs legitimate   53.58  (ratio 1.02x)
scenario 3: mean  265.72  vs legitimate   53.58  (ratio 4.96x)


## 2. Features, and the label delay

Terminal risk features use labels, so they carry a seven-day delay: the window
ends seven days before the transaction being scored. Customer features use no
labels and carry no delay.

The cell below demonstrates the property directly — mark one transaction as
fraudulent and watch when it becomes visible.

In [4]:
from rtml.config import FeatureConfig

base = pd.Timestamp("2025-01-01")
n = 40
probe = pd.DataFrame({
    "transaction_id": np.arange(n),
    "customer_id": np.zeros(n, dtype=int),
    "terminal_id": np.zeros(n, dtype=int),
    "tx_datetime": [base + pd.Timedelta(days=i) for i in range(n)],
    "tx_amount": np.full(n, 50.0),
    "is_fraud": np.zeros(n, dtype=int),
})

for delay in (0, 3, 7, 14):
    fc = FeatureConfig(customer_windows=[7], terminal_windows=[7], risk_delay_days=delay)
    before = build_features(probe, fc)["terminal_risk_7d"].to_numpy()
    marked = probe.copy()
    marked.loc[20, "is_fraud"] = 1
    after = build_features(marked, fc)["terminal_risk_7d"].to_numpy()
    changed = np.flatnonzero(np.abs(after - before) > 1e-9)
    print(f"delay {delay:2d}d -> fraud on day 20 first visible on day {changed.min()} "
          f"(expected >= {20 + delay})")

delay  0d -> fraud on day 20 first visible on day 20 (expected >= 20)
delay  3d -> fraud on day 20 first visible on day 23 (expected >= 23)
delay  7d -> fraud on day 20 first visible on day 27 (expected >= 27)
delay 14d -> fraud on day 20 first visible on day 34 (expected >= 34)


In [5]:
featured = build_features(tx, config.features)
names = feature_names(config.features)
featured[names].describe().T[["mean", "std", "min", "max"]].round(3)

,mean,std,min,max
tx_amount,56.750,51.636,0.00,1007.5
tx_during_weekend,0.276,0.447,0.00,1.0
tx_during_night,0.132,0.338,0.00,1.0
customer_nb_tx_1d,3.685,1.909,1.00,15.0
customer_avg_amount_1d,56.717,41.141,0.01,876.4
customer_nb_tx_7d,19.072,8.436,1.00,50.0
customer_avg_amount_7d,56.674,34.511,0.21,842.4
customer_nb_tx_30d,66.856,35.917,1.00,161.0
customer_avg_amount_30d,56.540,31.716,0.21,842.4
terminal_nb_tx_1d,1.170,1.259,0.00,11.0


## 3. Which features carry the signal

Separation between the fraud and legitimate distributions, per feature. The
terminal risk features separate strongly, which is scenario 2 becoming
detectable; the customer amount ratio is scenario 3.

In [6]:
rows = []
for name in names:
    a = featured.loc[featured.is_fraud == 1, name]
    b = featured.loc[featured.is_fraud == 0, name]
    pooled = np.sqrt((a.var() + b.var()) / 2) or 1e-9
    rows.append({
        "feature": name,
        "fraud_mean": a.mean(),
        "legit_mean": b.mean(),
        # Standardised mean difference: comparable across features with
        # wildly different units.
        "separation": abs(a.mean() - b.mean()) / pooled,
    })
pd.DataFrame(rows).sort_values("separation", ascending=False).round(3).to_string(index=False)

'                feature  fraud_mean  legit_mean  separation\n       terminal_risk_7d       0.404       0.021       1.166\n       terminal_risk_1d       0.336       0.013       0.956\n      terminal_risk_30d       0.200       0.025       0.884\n              tx_amount     124.841      53.578       0.656\n customer_avg_amount_1d      96.068      54.883       0.566\n customer_avg_amount_7d      77.649      55.696       0.462\n     terminal_nb_tx_30d      35.132      27.774       0.347\n      terminal_nb_tx_7d       9.250       7.791       0.270\ncustomer_avg_amount_30d      64.935      56.149       0.244\n     customer_nb_tx_30d      74.683      66.491       0.236\n      terminal_nb_tx_1d       1.339       1.162       0.139\n      customer_nb_tx_7d      19.928      19.033       0.108\n      customer_nb_tx_1d       3.733       3.683       0.026\n      tx_during_weekend       0.272       0.276       0.010\n        tx_during_night       0.132       0.132       0.001'

## 4. Models, and why PR-AUC leads

At a fraud rate near 1%, ROC-AUC is dominated by the vast negative class:
thousands of extra false positives barely move it. Watch how little ROC-AUC
separates these models compared with PR-AUC.

In [7]:
models, split = train_all(featured, config)
comparison = pd.DataFrame([{
    "model": m.name,
    "PR_AUC": m.test_metrics["pr_auc"],
    "ROC_AUC": m.test_metrics["roc_auc"],
    "precision": m.test_metrics["precision"],
    "recall": m.test_metrics["recall"],
    "card_P_at_100": m.test_metrics.get("card_precision_at_100", float("nan")),
    "p95_latency_ms": m.latency.get("latency_p95_ms", float("nan")),
    "fit_seconds": m.fit_seconds,
} for m in models]).round(4)
comparison

,model,PR_AUC,ROC_AUC,precision,recall,card_P_at_100,p95_latency_ms,fit_seconds
0,lightgbm,0.7332,0.8770,0.7648,0.6955,0.5053,0.8945,0.4095
1,xgboost,0.7235,0.8733,0.7639,0.6806,0.4947,0.6327,0.4957
2,logistic_regression,0.6201,0.8856,0.6597,0.6686,0.5059,0.4916,0.7527
3,random_forest,0.6029,0.8728,0.6967,0.6622,0.5059,38.2364,1.8475
4,isolation_forest,0.4512,0.8584,0.5757,0.5227,0.4529,9.4097,0.2770


In [8]:
spread_pr = comparison["PR_AUC"].max() - comparison["PR_AUC"].min()
spread_roc = comparison["ROC_AUC"].max() - comparison["ROC_AUC"].min()
print(f"PR-AUC spread across models:  {spread_pr:.4f}")
print(f"ROC-AUC spread across models: {spread_roc:.4f}")
print(f"\nPR-AUC separates them {spread_pr/spread_roc:.1f}x more than ROC-AUC does.")
print("\nRanking by ROC-AUC would pick:", comparison.loc[comparison.ROC_AUC.idxmax(), "model"])
print("Ranking by PR-AUC picks:       ", comparison.loc[comparison.PR_AUC.idxmax(), "model"])

PR-AUC spread across models:  0.2820
ROC-AUC spread across models: 0.0272

PR-AUC separates them 10.4x more than ROC-AUC does.

Ranking by ROC-AUC would pick: logistic_regression
Ranking by PR-AUC picks:        lightgbm


## 5. The operating point

A probability model is not a decision until a threshold makes it one. The
threshold is chosen on validation, never on test — choosing it on the evaluation
window is the most common way a reported precision turns out to be unreachable.

In [9]:
from rtml.evaluation.metrics import precision_recall_table, threshold_metrics

best = models[0]
X_test = split.test[best.features].to_numpy(dtype=np.float32)
scores = best.predict_proba(X_test)
labels = split.test["is_fraud"].to_numpy()

print(f"threshold chosen on validation: {best.threshold:.4f}\n")
curve = precision_recall_table(labels, scores, points=12)
print(curve.round(4).to_string(index=False))

threshold chosen on validation: 0.9066

 threshold  precision  recall
    0.0003     0.0524  1.0000
    0.0165     0.0557  0.9738
    0.0293     0.0606  0.9639
    0.0429     0.0659  0.9433
    0.0581     0.0725  0.9228
    0.0744     0.0808  0.8994
    0.0931     0.0919  0.8768
    0.1154     0.1074  0.8541
    0.1451     0.1306  0.8314
    0.1862     0.1698  0.8109
    0.2518     0.2459  0.7833
    0.4535     0.4760  0.7599
    0.9999     1.0000  0.0064


In [10]:
# What each operating point costs an investigation team.
for threshold in (0.5, best.threshold, 0.99):
    m = threshold_metrics(labels, scores, threshold)
    print(f"threshold {threshold:.4f}: precision {m['precision']:.3f}  recall {m['recall']:.3f}  "
          f"{int(m['alerts']):,} alerts ({m['alert_rate']:.2%} of traffic), "
          f"{int(m['false_positives']):,} wasted investigations")

threshold 0.5000: precision 0.515  recall 0.755  2,070 alerts (7.68% of traffic), 1,004 wasted investigations
threshold 0.9066: precision 0.765  recall 0.695  1,284 alerts (4.77% of traffic), 302 wasted investigations


threshold 0.9900: precision 0.913  recall 0.538  831 alerts (3.08% of traffic), 72 wasted investigations


## 6. Per-scenario recall

The aggregate hides which pattern a model misses, and the answer is rarely the
one you would guess. Scenario 1 is a deterministic rule — every amount above
220 is fraud — and the model learns it almost perfectly, yet it is often the
*worst* served at the deployed threshold, because a single F1-optimal cut-off
can land above that scenario's median score. The model is not the problem
there; the operating point is. `docs/results.md` §5 quantifies this at full
scale.


In [11]:
from rtml.evaluation.metrics import recall_by_group

per_scenario = recall_by_group(
    labels, scores, split.test["fraud_scenario"].to_numpy(), best.threshold
)
comparison = pd.DataFrame(
    {
        scenario: {
            "frauds": int(stats["frauds"]),
            f"recall@{best.threshold:.3f}": round(stats["recall"], 3),
            "recall@0.5": round(
                recall_by_group(
                    labels, scores, split.test["fraud_scenario"].to_numpy(), 0.5
                )[scenario]["recall"],
                3,
            ),
        }
        for scenario, stats in sorted(per_scenario.items())
    }
).T
comparison.index.name = "scenario"
comparison


,frauds,recall@0.907,recall@0.5
scenario,,,
1,16.0,0.625,0.812
2,1030.0,0.661,0.707
3,366.0,0.795,0.888


These figures come from a small configuration so the notebook stays quick. The
full-scale numbers — 1.8 million transactions, 556,547 test rows, 4,640 frauds —
are in `docs/results.md`, along with every limitation.